In [ ]:
from pathlib import Path
import json
import warnings
import sys
from typing import List

import numpy as np
import pandas as pd
import h5py
from scipy import stats

from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

In [ ]:
repo_dir = Path("../../..")

In [ ]:
if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))
    
from analysis.curve_fitting.src.fitting_functions import LOSS_FUNCTIONS
from analysis.curve_fitting.src.utils import apply_filters, load_yaml, convert_loss_parameters, convert_loss_parameters_batch

from visualization.src.utils import set_ticks, save_figs, apply_hiearchical_aggregation
from visualization.src.utils import BENCHMARK_NAME_MAPPING, BENCHMARK_COLORS, METRICS_COMPACT, GENERIC_GROUPING_COLUMNS, ARCHITECUTURE_FAMILY_COLORS
from visualization.src.visualize import plot_reg, plot_reg_bivariate, plot_confidence_intervals

In [ ]:
sns.set_theme(style='whitegrid')
sns.set_theme(style='ticks')

In [ ]:
artifacts_dir = repo_dir / "extract_n_eval" / "artifacts"
save_dir = repo_dir / "visualization" / "paper" / "main" / "figures"

assert artifacts_dir.exists(), f"artifacts directory {artifacts_dir} does not exist!"

In [ ]:
results_file = artifacts_dir / "finetuning_results.csv"

df_all = pd.read_csv(results_file)

In [ ]:
df_all.columns

In [ ]:
benchmark_order = {
    k: i for i, k in enumerate(BENCHMARK_NAME_MAPPING.keys())
}

In [ ]:
def get_base_model_id(model_id: str) -> str:
    arch_map = {
        'deit_small': 'deit_small_imagenet_full_seed-0',
        'convnext_small': 'convnext_small_imagenet_full_seed-0',
        'resnet50': 'resnet50_imagenet_full',
    }
    return arch_map[model_id.split('-')[0]]

In [ ]:
GROUPBY_COLS = ['model_id', 'benchmark_name', 'seed', 'finetuning_dataset']

def filter_df(df_orig: pd.DataFrame, exp_type:str, groupby_cols: List[str]=GROUPBY_COLS, data_pct:int=None, model_id:str=None, set_base_model_id:bool=True) -> pd.DataFrame:
    df = df_orig.copy()
    df = df[df['exp_type'] == exp_type]
    if data_pct is not None:
        df = df[df['data_pct'] == data_pct]
    if model_id is not None:
        df = df[df['model_id'] == model_id]
        
    df = apply_hiearchical_aggregation(
        df = df,
        groupby_cols = groupby_cols,
        agg_cols = METRICS_COMPACT + ['task_evaluation_acc'],
        agg_func = 'mean'
    )
    
    if set_base_model_id:
        df['base_model_id'] = df['model_id'].apply(get_base_model_id)
    
    return df

In [ ]:
LEFT_ON_COLS = ['base_model_id', 'benchmark_name']
RIGHT_ON_COLS = ['model_id', 'benchmark_name']
SUFFIXES = ('_finetuned', '_baseline')
SORT_BY_COLS = ['finetuning_dataset', 'benchmark_name']

def get_df_diffs(df1: pd.DataFrame, df2: pd.DataFrame, left_on_cols: List[str]=LEFT_ON_COLS, right_on_cols: List[str]=RIGHT_ON_COLS, suffixes: tuple=SUFFIXES, sort_by_cols: List[str]=SORT_BY_COLS) -> pd.DataFrame:
    df_merged = df1.merge(
        right=df2, 
        left_on=left_on_cols, 
        right_on=right_on_cols, 
        suffixes=suffixes
    )
    
    for metric in METRICS_COMPACT + ['task_evaluation_acc']:
        df_merged[f'{metric}_diff'] = df_merged[f'{metric}_finetuned'] - df_merged[f'{metric}_baseline']
    
    sorting_cols = []
    if 'finetuning_dataset' in sort_by_cols:
        sorting_cols.append('finetuning_dataset_order')
        df_merged['finetuning_dataset_order'] = df_merged['finetuning_dataset'].map(benchmark_order)
    if 'benchmark_name' in sort_by_cols:
        sorting_cols.append('benchmark_order')
        df_merged['benchmark_order'] = df_merged['benchmark_name'].map(benchmark_order)

    if sorting_cols:
        df_merged = df_merged.sort_values(
            by=sorting_cols
        )
        
    return df_merged

# Fine-Tuning Impact Across Benchmarks

In [ ]:
df_finetuned = filter_df(
    df_orig=df_all, 
    exp_type='finetuning_data_pct',
    data_pct=100
)
df_finetuned.shape

In [ ]:
df_baseline = filter_df(
    df_orig=df_all, 
    groupby_cols=['model_id', 'benchmark_name'],
    exp_type='finetuning_baseline',
    model_id='deit_small_imagenet_full_seed-0',
    set_base_model_id=False
)
df_baseline.shape

In [ ]:
df_diff_ds = get_df_diffs(
    df1=df_finetuned, 
    df2=df_baseline
)
df_diff_ds.shape

In [ ]:
zoom = 0.75
zoom = 0.6
figsize = (15,  9)
figsize = (figsize[0] * zoom, figsize[1] * zoom)
fig, axes = plt.subplots(
    2, 3, 
    figsize=figsize, 
    dpi=300,
    # sharex=True,
    sharey=True,
    
)

for i, ds in enumerate(df_diff_ds['finetuning_dataset'].unique()):
    ax = axes.flatten()[i]
    
    df_subset = df_diff_ds[df_diff_ds['finetuning_dataset'] == ds].copy()
    df_subset.benchmark_name = df_subset.benchmark_name.map(BENCHMARK_NAME_MAPPING)
    
    sns.barplot(
        data=df_subset,
        x='benchmark_name',
        y='pearsonr_nc_diff',
        hue='benchmark_name',
        palette={v: BENCHMARK_COLORS[k] for k,v in BENCHMARK_NAME_MAPPING.items()},
        ax=ax
    )
    # ax.axhline(0, color='red', linestyle='--')
    
    
    # ax.set_title(f'Finetuned on: {benchmark_names[ds]}', fontsize=16, fontweight='bold')
    ax.set_title(f'{BENCHMARK_NAME_MAPPING[ds]}', fontsize=16, fontweight='bold')
    # ax.set_ylabel(r"$\boldsymbol{\Delta}$ Average Alignment", fontsize=16, fontweight='bold')
    ax.set_ylabel(r"$\Delta$ Average Alignment", fontsize=16, fontweight='bold')
    ax.set_xlabel(f"Testing Benchmark", fontsize=12, fontweight='bold', alpha=0.7)
    
    if  i < 3:
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
        # ax.set_xticklabels([])
        # ax.set_yticklabels([])
        ax.set_ylabel(None)
        ax.set_xlabel(None)
    
    else:
        # Rotate x labels for better readability
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
        # ax.set_xlabel(None)
    
    ax.set_ylim(-0.007, 0.017)
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle("Finetuning Data", fontsize=16, fontweight='bold', y=1.02, alpha=0.7)
plt.suptitle("Finetuning:", fontsize=12, fontweight='bold', y=1.02)
plt.suptitle("Finetuning Data", fontsize=12, fontweight='bold', y=1.02, alpha=0.7)
plt.tight_layout()


# figures_dir = save_dir
# fig_name = 'fig5_finetuning_diffs'
# formats = ['pdf', 'png', 'svg']
# save_figs(fig, figures_dir, fig_name, formats=formats)

figures_dir = save_dir
fig_name = 'fig5-finetuning-improvements-per_benchmark'
formats = ['pdf', 'png', 'svg']
save_figs(fig=fig, save_dir=figures_dir, base_filename=fig_name, dpi=300, formats=formats)